# Day 2 — Preprocessing & Feature Engineering
**sem5_paper — Ruchit**

Pipeline: drop leakage columns → encode `disk` → stratified 80/20 split (`random_state=42`) → scale numeric features (fit on train only) → save splits.

Outlier handling is *investigated but not applied by default* — see Section 5 for why.

## 1. Load raw data

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RAW_PATH = Path("/Users/ruchitbhalerao/Desktop/files/Data.csv")
OUT_DIR = Path("/Users/ruchitbhalerao/Desktop/data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

df = pd.read_csv(RAW_PATH)
print(df.shape)
df.head()

(100000, 63)


,timestamp,CPU (%),CPU system (%),memory (GB),threads,disk,ops_sda,util_sda (%),available (MB),writeback (MB),...,Network Port Utilization,Disk Write Operations per Second,Network Interface Throughput,CPU Utilization by Core,CPU Load Average,Disk I/O Utilization by Process,Memory Usage by Process,Network Interface Broadcast Packets,Network Interface Multicast Packets,TCP Connection Count by State
0,1653264015,12,8,2,137,sda2,203,87,601,326,...,21.374984,27.442524,773.778802,82.276509,1.302460,0.907449,31.989304,4.342209,75.032005,80.010555
1,1653264465,29,6,5,182,sda2,160,64,599,156,...,41.528810,2.687175,862.055106,3.413034,4.815457,5.595459,46.555521,62.916891,12.973314,60.718622
2,1653264668,10,9,6,246,sda2,397,90,1000,276,...,5.112373,50.676002,296.277126,53.977365,1.945296,17.889892,81.992400,93.262324,1.842360,69.952761
3,1653264352,31,13,8,276,sda1,433,74,350,704,...,95.334200,39.203402,401.590840,246.671971,0.069329,5.262651,99.036962,73.603123,73.362418,252.569617
4,1653264755,21,14,3,113,sda1,359,82,110,123,...,29.856904,86.123367,793.165660,8.777634,1.437517,88.223138,26.520183,29.165669,23.225988,78.667086


## 2. Drop target / leakage / raw timestamp

| Column | Action | Reason |
|---|---|---|
| `is_attack` | move to `y` | target |
| `attack_name` | drop | target leakage (only populated on attack rows) |
| `timestamp` | drop | raw epoch, not a useful direct measurement as-is |

In [3]:
y = df["is_attack"]
X = df.drop(columns=["is_attack", "attack_name", "timestamp"])

print("X shape:", X.shape)
print("Class balance:")
print(y.value_counts(normalize=True))

X shape: (100000, 60)
Class balance:
is_attack
0    0.9
1    0.1
Name: proportion, dtype: float64


## 3. Data quality check (nulls, duplicates)
Confirming there's nothing left to clean before splitting.

In [4]:
print("Nulls per column (X):")
print(X.isnull().sum()[X.isnull().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())

Nulls per column (X):
Series([], dtype: int64)

Duplicate rows: 0


## 4. Encode categorical column `disk`
`disk` is nominal (`sda1`, `sda2`, `sda5` — no inherent order), so it's **one-hot encoded** rather than label-encoded, to avoid implying a false ordinal relationship.

In [5]:
print(X["disk"].value_counts())

X = pd.get_dummies(X, columns=["disk"], prefix="disk")
X.head()

disk
sda2    33589
sda5    33206
sda1    33205
Name: count, dtype: int64


,CPU (%),CPU system (%),memory (GB),threads,ops_sda,util_sda (%),available (MB),writeback (MB),cpu_intr,io (MB/s),...,CPU Utilization by Core,CPU Load Average,Disk I/O Utilization by Process,Memory Usage by Process,Network Interface Broadcast Packets,Network Interface Multicast Packets,TCP Connection Count by State,disk_sda1,disk_sda2,disk_sda5
0,12,8,2,137,203,87,601,326,74,3,...,82.276509,1.302460,0.907449,31.989304,4.342209,75.032005,80.010555,False,True,False
1,29,6,5,182,160,64,599,156,40,4,...,3.413034,4.815457,5.595459,46.555521,62.916891,12.973314,60.718622,False,True,False
2,10,9,6,246,397,90,1000,276,38,2,...,53.977365,1.945296,17.889892,81.992400,93.262324,1.842360,69.952761,False,True,False
3,31,13,8,276,433,74,350,704,86,8,...,246.671971,0.069329,5.262651,99.036962,73.603123,73.362418,252.569617,True,False,False
4,21,14,3,113,359,82,110,123,57,3,...,8.777634,1.437517,88.223138,26.520183,29.165669,23.225988,78.667086,True,False,False


## 5. Outlier check — investigate before deciding

Before blanket-removing outliers, check whether they correlate with `is_attack`. If attacks *are* the extreme values (which is typical for intrusion-detection data — a spike in load, memory, or errors is often the attack signature), removing them would strip out the minority class and destroy the signal the model needs to learn.

In [6]:
from scipy import stats

numeric_cols_check = X.select_dtypes(include=["int64", "float64"]).columns
numeric_cols_check = [c for c in numeric_cols_check if not c.startswith("disk_")]

outlier_summary = []
for col in numeric_cols_check:
    q1, q3 = X[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (X[col] < lo) | (X[col] > hi)
    if mask.sum() > 0:
        outlier_summary.append({
            "column": col,
            "n_outliers": int(mask.sum()),
            "attack_rate_in_outliers": round(y[mask].mean(), 3),
            "attack_rate_overall": round(y.mean(), 3),
        })

outlier_df = pd.DataFrame(outlier_summary).sort_values("n_outliers", ascending=False)
outlier_df.head(15)

,column,n_outliers,attack_rate_in_outliers,attack_rate_overall
2,available (MB),12860,0.097,0.1
5,load,4637,1.000,0.1
8,ram (GB),4553,1.000,0.1
13,network_errors,3720,1.000,0.1
32,Database Read Operations per Second,2568,1.000,0.1
15,process_memory_usage,2485,1.000,0.1
33,Database Write Operations per Second,2462,1.000,0.1
9,network_bandwidth,2400,1.000,0.1
14,process_cpu_usage,2204,1.000,0.1
4,io (MB/s),1998,1.000,0.1


**Finding:** for several features (`load`, `ram (GB)`, `network_errors`, `process_cpu_usage`, etc.), close to 100% of IQR-flagged outlier rows are attack samples — vs. a ~10% base rate. These "outliers" are the signal, not noise.

**Decision: outliers are NOT removed.** Removing them would delete most of the minority class and worsen the imbalance. This is a deliberate, documented design choice — reviewers in the security/anomaly-detection space will expect to see this reasoning rather than a default IQR-strip.

If a genuinely *noisy* (non-attack) outlier needs handling later, the safer version of this step is to check/clip outliers only within `is_attack == 0` rows, and only on the training set — not attempted here by default. Uncomment the cell below if the team decides to apply it.

In [7]:
# Optional / not applied by default — outlier clipping restricted to the NORMAL class only,
# and only ever fit on the training set (after the split below), to avoid touching attack signal
# or leaking test-set information.
#
# def clip_outliers_normal_class(X_train, y_train, cols):
#     X_train = X_train.copy()
#     normal_mask = (y_train == 0)
#     for col in cols:
#         q1, q3 = X_train.loc[normal_mask, col].quantile([0.25, 0.75])
#         iqr = q3 - q1
#         lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
#         X_train[col] = X_train[col].clip(lower=lo, upper=hi)
#     return X_train
#
# X_train = clip_outliers_normal_class(X_train, y_train, numeric_cols)

## 6. Stratified 80/20 train/test split
`random_state=42`, stratified on `is_attack` so both splits keep the 90/10 class balance. Done **before** scaling so the scaler only ever sees training data.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("\ny_train balance:\n", y_train.value_counts(normalize=True))
print("\ny_test balance:\n", y_test.value_counts(normalize=True))

X_train: (80000, 62)  X_test: (20000, 62)

y_train balance:
 is_attack
0    0.9
1    0.1
Name: proportion, dtype: float64

y_test balance:
 is_attack
0    0.9
1    0.1
Name: proportion, dtype: float64


## 7. Scale numeric features

**StandardScaler** chosen over MinMaxScaler: these system metrics (CPU, latency, throughput, etc.) are continuous with genuine spikes (see Section 5) rather than naturally bounded ranges — standardization is less distorted by those extremes than min-max scaling would be.

Fit on `X_train` only, applied to both. One-hot `disk_*` columns are already 0/1 and left unscaled.

In [9]:
numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
numeric_cols = [c for c in numeric_cols if not c.startswith("disk_")]

X_train[numeric_cols] = X_train[numeric_cols].astype("float64")
X_test[numeric_cols] = X_test[numeric_cols].astype("float64")

scaler = StandardScaler()
X_train.loc[:, numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test.loc[:, numeric_cols] = scaler.transform(X_test[numeric_cols])

X_train.head()

,CPU (%),CPU system (%),memory (GB),threads,ops_sda,util_sda (%),available (MB),writeback (MB),cpu_intr,io (MB/s),...,CPU Utilization by Core,CPU Load Average,Disk I/O Utilization by Process,Memory Usage by Process,Network Interface Broadcast Packets,Network Interface Multicast Packets,TCP Connection Count by State,disk_sda1,disk_sda2,disk_sda5
12068,0.946703,0.683173,0.514522,1.249257,-1.640065,-0.828716,1.412970,-1.000661,-0.215731,0.731770,...,0.669723,0.597195,-0.975570,-0.780459,0.099774,-1.613002,1.132148,False,False,True
19985,1.486776,2.856174,0.514522,0.319241,-0.652154,1.801090,-0.533787,0.371201,0.060138,2.014900,...,0.999134,1.355760,-0.101867,-1.256447,-0.632355,0.693527,-1.312041,False,True,False
95058,0.082588,-0.403328,-0.322533,0.622037,1.449250,0.047886,-0.591819,-0.633853,1.053264,0.731770,...,-0.418113,1.188368,-0.700229,-0.457856,1.113294,-0.792700,0.393785,True,False,False
37517,-0.241456,0.031272,1.351576,-1.064969,1.139482,-0.086976,-0.018687,-0.699878,1.494654,0.090204,...,0.750476,-0.725409,1.293987,0.768537,-0.356170,-0.315479,-1.146250,False,False,True
25645,0.730675,-0.620628,0.095994,-0.632403,0.486456,0.857057,-0.576206,-0.032288,0.667049,0.731770,...,-0.612116,-0.240515,0.376400,-0.024038,-0.576826,-1.357375,-1.365412,False,True,False


## 8. Class imbalance strategy

90/10 split is moderate, not extreme. Planned approach:
- **Primary:** class weighting (`class_weight='balanced'` or equivalent) applied per model at training time.
- **Fallback:** SMOTE on the training set only, if a model underperforms with class weighting alone — **never applied to `X_test`/`y_test`**.

Confirm this matches Siya's class-balance numbers before finalizing for the Methodology section.

## 9. Save processed splits

In [10]:
X_train.to_csv(OUT_DIR / "X_train.csv", index=False)
X_test.to_csv(OUT_DIR / "X_test.csv", index=False)
y_train.to_csv(OUT_DIR / "y_train.csv", index=False)
y_test.to_csv(OUT_DIR / "y_test.csv", index=False)

print(f"Saved X_train, X_test, y_train, y_test to {OUT_DIR}/")

Saved X_train, X_test, y_train, y_test to /Users/ruchitbhalerao/Desktop/data/
